# P4. 법령 Q&A 챗봇

국가법령정보 공동활용 API 로 개인정보 보호법 원문(현행 · 시행예정 두 판)을 받아 조문(항 · 호) 단위로 자르고, ko-sroberta 임베딩과 Qdrant 로 「조문 번호를 출처로 답하는」 검색을 만듭니다. 고정 길이로 자른 것과 나란히 놓고 무엇이 달라지는지 숫자로 확인합니다.

- 단계: 개별 프로젝트 · 비정형 · 예상 6~8시간
- 실행 환경: Google Colab(브라우저만 있으면 됩니다)
- 데이터: 법제처 국가법령정보 공동활용(DRF Open API) — 개인정보 보호법 현행판(법률 제21445호, 2026.3.10 공포 · 2026.9.11 시행)과 시행예정판(법률 제21910호, 2026.9.8 공포 · 2027.3.9 시행) 원문, 2026-10-28 받은 스냅샷. 법령은 저작권법 제7조에 따라 보호받지 않는 저작물

**빈칸 노트북** — 코드 속 `___①___` 자리 16곳을 채웁니다. 줄 끝 주석이 힌트입니다.
막히면 [따라하기 노트북](https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/P4/P4_guide.ipynb)의 같은 단계를 보세요. 정답은 맨 아래에 있습니다.

「개인정보를 다른 곳에 넘겨도 되나요?」 같은 질문에 법령 챗봇이 답한다면, 답보다 먼저 **「몇 조 몇 항에 그렇게 쓰여 있는가」**가 나와야 합니다.
근거 조문을 댈 수 없는 법률 답변은 믿고 쓸 수 없기 때문입니다.

이 프로젝트에서는 법령 원문을 국가법령정보 API 로 직접 받아, 7주차에 배운 RAG 흐름을 **법령에 맞게** 다시 짭니다.

```text
DRF API(JSON) → 조문 · 항 · 호 단위 청크(+조문 번호 메타데이터) → ko-sroberta 임베딩 → Qdrant → 검색 1위 조문 → (키가 있으면) gpt-4o-mini 답 + 조문 인용
```

다 끝내면 다음 세 가지를 말할 수 있습니다.

- 법령 API 에서 **현행판과 시행예정판**이 어떻게 다르게 나오고, 왜 둘을 구분해야 하는지
- 조문 단위로 자르면 고정 길이(500자)로 자를 때보다 출처를 얼마나 더 정확히 댈 수 있는지
- 검색 결과에 조문 번호를 붙여 LLM 에 넘기는 방법

## 1단계 · 준비 — 라이브러리와 데이터

임베딩은 키 없이 도는 `jhgan/ko-sroberta-multitask`(768차원)를, 벡터 DB 는 Qdrant 를 메모리 모드(`:memory:`)로 씁니다. 7주차 7-3장에서는 Docker 로 Qdrant 서버를 띄웠는데, Colab 에서는 서버 없이 같은 API 를 그대로 쓸 수 있습니다.

데이터는 두 갈래입니다.
2단계에서 법령 API 를 직접 불러 보고, 그 뒤 단계는 2026-10-28 에 같은 API 로 받아 둔 스냅샷으로 진행합니다. 법령은 개정되면 조문이 바뀌므로, 스냅샷을 써야 누가 언제 돌려도 확인 포인트 숫자가 같게 나옵니다.

- `law_pipa_20260911.json.gz` — 개인정보 보호법 **현행판**(법률 제21445호, 2026.3.10 공포, 2026.9.11 시행)
- `law_pipa_20270309.json.gz` — 같은 법의 **시행예정판**(법률 제21910호, 2026.9.8 공포, 2027.3.9 시행)

법령은 저작권법 제7조(보호받지 못하는 저작물)에 해당해 자유롭게 이용할 수 있습니다. 다만 어느 시점의 판인지는 반드시 함께 적어 둡니다.

In [ ]:
import sys, subprocess, importlib.util
def need(mod, pkg=None):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg or mod], check=True)
need("qdrant_client", "qdrant-client")
need("sentence_transformers", "sentence-transformers")

import os, re, json, gzip, shutil, urllib.request
import pandas as pd

DATA = "https://raw.githubusercontent.com/aebonlee/materials/main/data-eng/projects/data/"

def fetch(name):
    """프로젝트 데이터를 ./data 폴더에 한 번만 받아 둡니다."""
    os.makedirs("data", exist_ok=True)
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        local = os.environ.get("DE_DATA_LOCAL")          # 오프라인 실습용 — 자료 폴더가 내 컴퓨터에 있을 때
        if local:
            shutil.copy(os.path.join(local, name), dst)
        else:
            urllib.request.urlretrieve(DATA + name, dst)
    return dst

for name in ["law_pipa_20260911.json.gz", "law_pipa_20270309.json.gz"]:
    p = fetch(name)
    print(p, round(os.path.getsize(p) / 1e3), "KB")

> **확인 포인트** — 두 파일이 각각 52KB 쯤으로 받아집니다(압축을 풀면 약 400KB 의 JSON).

## 2단계 · 법령 API 직접 불러 보기 — 현행과 시행예정

국가법령정보 공동활용(open.law.go.kr)의 DRF API 는 두 가지 주소를 씁니다.

- `lawSearch.do` — 법령 이름으로 **목록**을 찾습니다. 결과에 법령일련번호(MST) · 공포일자 · 시행일자가 있습니다.
- `lawService.do` — MST 로 **본문**(조문 전체)을 받습니다.

`OC` 는 신청자 아이디(이메일 앞부분)입니다. 실습에서는 공개 시험값 `test` 로 부르지만, 프로젝트를 계속 쓰려면 open.law.go.kr 에서 OC 를 신청해 바꾸세요.

중요한 것은 `nw` 파라미터입니다. 기본값은 **현행** 법령만 보여 줍니다. 이미 공포됐지만 아직 시행 전인 개정은 `nw=2`(시행예정)를 붙여야 보입니다.
현행판만 보고 「개정 소식이 없다」고 답하면 몇 달 뒤에는 틀린 답이 됩니다.

In [ ]:
UA = {"User-Agent": "Mozilla/5.0 (Macintosh)"}
DRF = "https://www.law.go.kr/DRF/"

def law_search(query, nw=None, oc="test"):
    params = {"OC": oc, "target": "law", "type": "JSON", "query": query}
    if nw:
        params["___①___"] = nw   # ① 현행 · 시행예정을 고르는 파라미터
    url = DRF + "lawSearch.do?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers=UA)
    d = json.load(urllib.request.urlopen(req, timeout=20))["LawSearch"]
    laws = d.get("law", [])
    laws = laws if isinstance(laws, list) else [laws]           # 한 건이면 목록이 아니라 dict 하나로 온다
    return pd.DataFrame(laws)[["법령명한글", "현행연혁코드", "법령일련번호", "공포일자", "시행일자"]]

import urllib.parse
try:
    print(law_search("개인정보 보호법"))                      # 현행
    print(law_search("개인정보 보호법", nw=___②___))   # ② 시행예정 값
except Exception as e:
    print("API 를 부르지 못했습니다(스냅샷으로 계속합니다):", type(e).__name__, e)

> **확인 포인트** — 2026-10-28 에 부르면 현행 「개인정보 보호법」(MST 283839, 시행 20260911)과 시행령이 나오고, `nw=2` 로는 시행예정판(MST 289415, 공포 20260908, 시행 20270309)이 나옵니다.
> 날짜가 지나 시행예정판이 시행되면 결과가 바뀝니다. 그래서 다음 단계부터는 스냅샷으로 확인합니다.

일부러 한 번 틀려 봅니다. `OC` 를 빼면 API 는 오류 대신 **설명 문장이 든 JSON** 을 돌려줍니다. 상태 코드 200 이라고 성공한 것이 아닙니다.

In [ ]:
try:
    url = DRF + "lawSearch.do?" + urllib.parse.urlencode({"target": "law", "type": "JSON", "query": "개인정보 보호법"})
    d = json.load(urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=20))
    print(d)
    d["LawSearch"]
except KeyError as e:
    print("LawSearch 가 없습니다 — OC 를 빠뜨렸는지 확인하세요:", e)
except Exception as e:
    print("네트워크 오류:", type(e).__name__)

> **자주 겪는 오류** — 공공 API 는 실패해도 HTTP 200 에 안내 문장을 담아 주는 경우가 많습니다. 응답에 기대한 키(`LawSearch`, `법령`)가 있는지부터 확인하세요.

본문은 이렇게 받습니다(실행해 보면 스냅샷과 같은 구조의 JSON 이 옵니다). 다음 단계부터는 스냅샷을 읽습니다.

```text
https://www.law.go.kr/DRF/lawService.do?OC=test&target=law&type=JSON&MST=283839   ← 현행판
https://www.law.go.kr/DRF/lawService.do?OC=test&target=law&type=JSON&MST=289415   ← 시행예정판
```

## 3단계 · 본문 구조 살펴보기 — 조문 · 항 · 호 · 목

In [ ]:
def load_law(name):
    with gzip.open(os.path.join("data", name), "rt", encoding="utf-8") as f:
        return json.load(f)["___③___"]   # ③ 본문 JSON 의 맨 바깥 키

cur = load_law("law_pipa_20260911.json.gz")
nxt = load_law("law_pipa_20270309.json.gz")
for law in (cur, nxt):
    b = law["기본정보"]
    units = law["조문"]["조문단위"]
    arts = [u for u in units if u["조문여부"] == "___④___"]   # ④ 장 · 절 제목이 아닌 진짜 조문
    print(b["법령명_한글"], "법률 제" + b["공포번호"] + "호", "공포", b["공포일자"], "시행", b["시행일자"], "| 조문단위", len(units), "· 조문", len(arts))

`조문단위` 에는 「제1장 총칙」 같은 **장 · 절 제목**(조문여부 = 전문)도 섞여 있어서 조문만 골라 셉니다.

> **확인 포인트** — 현행판은 조문단위 141개 중 조문 127개, 시행예정판은 146개 중 131개입니다.

In [ ]:
assert len([u for u in cur["조문"]["조문단위"] if u["조문여부"] == "조문"]) == 127
assert len([u for u in nxt["조문"]["조문단위"] if u["조문여부"] == "조문"]) == 131

조문 하나를 열어 봅니다. 제15조(개인정보의 수집 · 이용)는 `항` 목록 안에 `호` 목록이 들어 있는 구조입니다.

In [ ]:
art15 = next(u for u in cur["조문"]["조문단위"] if u["조문여부"] == "조문" and u["조문번호"] == "15")
print(art15["조문내용"])
for h in art15["항"]:
    print(" ", h["항내용"][:60], "…")
    for ho in h.get("호", []):
        print("     ", ho["호내용"][:50])

여기서 조심할 점이 셋 있습니다.

- `항` 이 **목록일 때도, dict 하나일 때도, 아예 없을 때도** 있습니다(항이 하나뿐인 조문은 dict, 항 없이 본문만 있는 조문은 없음).
- 제2조(정의)처럼 항 번호 없이 호만 있는 조문이 있습니다.
- 「제7조의2」 같은 가지 조문은 `조문번호` 7 + `조문가지번호` 2 로 나뉘어 옵니다.

In [ ]:
from collections import Counter
Counter(type(u.get("항")).__name__ for u in cur["조문"]["조문단위"] if u["조문여부"] == "조문")

> **확인 포인트** — 현행판 조문 127개 중 항이 목록인 것 97개, dict 하나인 것 12개, 없는 것 18개입니다.

## 4단계 · 조문(항 · 호) 단위로 자르기

청크 하나 = **항 하나**를 기본으로 합니다. 항 안에 호가 길게 이어져 400자를 넘으면 **호 하나씩** 다시 자릅니다.
자른 조각마다 앞에 `제15조(개인정보의 수집ㆍ이용)` 머리를 붙이고, 조문 번호를 `id` 메타데이터로 따로 둡니다. 머리를 붙이는 까닭은 조각만 따로 떼어 봐도 무슨 조문인지 알게 하기 위해서이고, `id` 를 따로 두는 까닭은 나중에 답에 출처로 붙이기 위해서입니다(7주차 7-2장 청킹 — 「의미 단위로 자르고 메타데이터를 남긴다」).

In [ ]:
CIRC = "①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳"

def as_list(x):
    """항 · 호 · 목이 dict 하나로 오거나 없을 때도 목록으로 맞춥니다."""
    if x is None:
        return []
    return x if isinstance(x, ___⑤___) else [x]   # ⑤ 이미 목록이면 그대로

def art_no(u):
    no = f"제{u['조문번호']}조"
    if u.get("___⑥___"):   # ⑥ 제7조의2 의 2
        no += f"의{u['조문가지번호']}"
    return no

def chunk_articles(law, limit=400):
    out = []
    for u in law["조문"]["조문단위"]:
        if u["조문여부"] != "조문":
            continue
        no, title, head = art_no(u), u.get("조문제목", ""), u["조문내용"].strip()
        hangs = as_list(u.get("항"))
        if not hangs:                                         # 항 없이 본문만 있는 조문
            if "삭제" not in head[:30]:                       # 「제12조 삭제 <2023.3.14>」 같은 빈 조문은 뺀다
                out.append({"id": no, "article": no, "title": title, "text": head})
            continue
        for h in hangs:
            hn = (h.get("항번호") or "").strip()
            hid = no + (f"제{CIRC.find(hn[:1]) + 1}항" if hn else "")
            lead = (h.get("항내용") or "").strip()
            if not hn and not lead:                           # 제2조 · 제59조처럼 항 번호 없이 호만 있으면 조문 첫 문장을 머리로
                lead = head.split(")", 1)[-1].strip()
            hos = as_list(h.get("___⑦___"))   # ⑦ 항 아래 단계
            body = [lead]
            for ho in hos:
                body.append(ho["호내용"].strip())
                body += [m["목내용"].strip() for m in as_list(ho.get("목"))]
            text = f"{no}({title}) " + "\n".join(b for b in body if b)
            if len(text) <= ___⑧___ or not hos:   # ⑧ 길이 기준
                out.append({"id": hid, "article": no, "title": title, "text": text})
                continue
            for ho in hos:                                    # 길면 호 하나씩
                hb = ho["호번호"].strip().rstrip(".") + (f"의{ho['호가지번호']}" if ho.get("호가지번호") else "")
                part = [ho["호내용"].strip()] + [m["목내용"].strip() for m in as_list(ho.get("목"))]
                out.append({"id": hid + f"제{hb}호", "article": no, "title": title,
                            "text": f"{no}({title}) {lead}\n" + "\n".join(part)})
    return out

cur_chunks = chunk_articles(cur)
nxt_chunks = chunk_articles(nxt)
print(len(cur_chunks), len(nxt_chunks))
pd.DataFrame(cur_chunks)[["id", "text"]].head(8)

> **확인 포인트** — 현행판 **605개**, 시행예정판 **629개** 조각이 나옵니다. 맨 위는 `제1조`, 그다음은 `제2조제1호`, `제2조제1의2호` … 처럼 정의 조문이 호 단위로 나뉩니다.

In [ ]:
assert len(cur_chunks) == 605 and len(nxt_chunks) == 629
ids = [c["id"] for c in cur_chunks]
assert len(ids) == len(set(ids))                            # 조각 id 가 겹치지 않아야 출처로 쓸 수 있다
print("가장 긴 조각:", max(len(c["text"]) for c in cur_chunks), "자")

> **Aebon's Tip** — ko-sroberta 는 한 번에 128토큰(한글 200~300자 안팎)까지만 봅니다. 그보다 긴 글은 **뒷부분을 읽지 않고 잘라 버립니다.**
> 제2조(정의)를 항 하나로 두면 2,400자가 넘어 뒤쪽 정의(가명정보 · 영상정보처리기기 등)가 검색되지 않습니다. 호 단위로 다시 자르는 이유가 이것입니다.

일부러 실패해 봅니다. `as_list` 없이 `항` 을 바로 돌리면 dict 하나로 온 조문에서 엉뚱하게 동작합니다.

In [ ]:
one = next(u for u in cur["조문"]["조문단위"] if isinstance(u.get("항"), dict))
try:
    for h in one["항"]:
        print(h["항내용"])
except TypeError as e:
    print(art_no(one), "— dict 를 그대로 돌리면 키 이름(글자)이 나옵니다:", e)

> **자주 겪는 오류** — dict 를 `for` 로 돌리면 값이 아니라 **키 이름**이 나옵니다. 그래서 `h["항내용"]` 이 글자에 대고 인덱싱을 하다 `TypeError` 가 납니다. 공공 API 의 「한 건이면 dict, 여러 건이면 list」 습관은 법령 API 말고도 흔합니다.

## 5단계 · 현행판과 시행예정판 비교

두 판을 조문 번호로 맞대 보면 시행예정판에서 무엇이 바뀌는지 바로 보입니다.

In [ ]:
def by_article(chunks):
    d = {}
    for c in chunks:
        d.setdefault(c["article"], []).append(c["text"])
    return {k: "\n".join(v) for k, v in d.items()}

A, B = by_article(cur_chunks), by_article(nxt_chunks)
added   = [k for k in B if k not in A]
changed = [k for k in A if k in B and A[k] != B[k]]
print("새로 생기는 조문:", added)
print("바뀌는 조문:", changed)
print(B[added[0]][:300])

> **확인 포인트** — 시행예정판에는 **제28조의12 ~ 제28조의15** 네 조문이 새로 생기고(인공지능기술 개발을 위한 개인정보 이용), 제2조 · 제7조의9 · 제37조의2 세 조문이 바뀝니다.

In [ ]:
assert added == ["제28조의12", "제28조의13", "제28조의14", "제28조의15"]
assert changed == ["제2조", "제7조의9", "제37조의2"]

> **Aebon's Tip** — 시행예정판의 `기본정보` 에는 `조문시행일자문자열` 이 있어, 같은 법 안에서도 조문마다 시행일이 다를 수 있음을 알려 줍니다(예: 현행판의 「20270701:제32조의2제1항 단서…」). 챗봇이 날짜를 다루려면 이 칸까지 메타데이터로 넣으세요.

## 6단계 · 임베딩하고 Qdrant 에 넣기

7주차 7-3장과 같은 순서입니다. 컬렉션을 만들고(차원 768, 코사인), 조각마다 벡터와 payload(조문 번호 · 제목 · 원문)를 함께 넣습니다.
판이 둘이므로 컬렉션도 둘로 나눕니다. 한 컬렉션에 섞으면 같은 조문 번호가 두 번 나와 어느 판의 내용인지 헷갈립니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

model = SentenceTransformer("jhgan/ko-sroberta-multitask")
qd = QdrantClient("___⑨___")   # ⑨ 서버 없이 메모리에

def build_index(name, chunks):
    vecs = model.___⑩___([c["text"] for c in chunks], batch_size=32, normalize_embeddings=True)   # ⑩ 글을 벡터로
    if qd.collection_exists(name):
        qd.delete_collection(name)
    qd.create_collection(name, vectors_config=models.VectorParams(size=___⑪___, distance=models.Distance.COSINE))   # ⑪ ko-sroberta 의 차원
    qd.upsert(name, [models.PointStruct(id=i, vector=v.tolist(), ___⑫___=c) for i, (v, c) in enumerate(zip(vecs, chunks))])   # ⑫ 벡터에 딸린 정보
    return qd.count(name).count

print("현행:", build_index("pipa_current", cur_chunks))
print("시행예정:", build_index("pipa_next", nxt_chunks))

> **확인 포인트** — `현행: 605` · `시행예정: 629` — 4단계의 조각 수와 같아야 빠진 것이 없습니다(Colab CPU 에서 1분 안쪽).

In [ ]:
assert qd.count("pipa_current").count == 605 and qd.count("pipa_next").count == 629

일부러 차원을 틀려 봅니다. 다른 모델(예: OpenAI text-embedding-3-small, 1536차원)로 만든 컬렉션에 ko-sroberta 벡터를 넣으면 이렇게 됩니다.

In [ ]:
qd.create_collection("wrong_dim", vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE))
try:
    qd.upsert("wrong_dim", [models.PointStruct(id=0, vector=model.encode("시험").tolist(), payload={})])
except Exception as e:
    print("넣지 못했습니다:", str(e)[:120])
qd.delete_collection("wrong_dim")

## 7단계 · 질문하고 1위 조문 확인하기

질문을 같은 모델로 벡터로 바꿔 가장 가까운 조각을 찾습니다. 결과의 `payload["id"]` 가 곧 출처입니다.

In [ ]:
def ask(q, coll="pipa_current", k=3):
    qv = model.encode(q, normalize_embeddings=True).tolist()
    hits = qd.___⑬___(coll, query=qv, limit=k).points   # ⑬ Qdrant 검색 메서드
    return [(h.payload["id"], round(h.score, 3), h.payload["text"]) for h in hits]

for q in ["개인정보가 유출되면 정보주체에게 알려야 하나요?", "만 14세 미만 아동의 개인정보를 처리하려면?"]:
    print("Q:", q)
    for cid, score, text in ask(q):
        print(f"   {score}  {cid:14s} {text[:50]}…")

> **확인 포인트** — 유출 질문의 1위는 **제34조(개인정보 유출 등의 통지 · 신고) 제1항**의 호, 아동 질문의 1위는 **제22조의2제1항**(아동의 개인정보 보호)입니다.

In [ ]:
assert ask("개인정보가 유출되면 정보주체에게 알려야 하나요?")[0][0].startswith("제34조제1항")
assert ask("만 14세 미만 아동의 개인정보를 처리하려면?")[0][0] == "제22조의2제1항"

이번에는 **같은 질문을 두 판에** 던져 봅니다. 시행예정판에만 있는 조문이 답이 되는 질문입니다.

In [ ]:
q = "인공지능 개발에 이미 수집한 개인정보를 써도 되나요?"
print("현행    1위:", ask(q, "pipa_current")[0][:2])
print("시행예정 1위:", ask(q, "___⑭___")[0][:2])   # ⑭ 시행예정판 컬렉션

> **확인 포인트** — 현행판에서는 관련 조문이 없어 제15조제3항(동의 없이 추가 이용)이 0.503 의 낮은 점수로 1위가 되고, 시행예정판에서는 **제28조의12제1항**이 1위로 나옵니다.
> 챗봇이 「2027년 3월 9일부터는 제28조의12 가 적용됩니다」라고 답하려면 두 판을 함께 두어야 합니다.

In [ ]:
assert ask(q, "pipa_current")[0][0] == "제15조제3항"
assert ask(q, "pipa_next")[0][0].startswith("제28조의12")

> **Aebon's Tip** — 1위 점수가 낮으면(여기서는 0.5 안팎) 「관련 조문을 찾지 못했다」고 답하게 하는 것이 안전합니다. 점수 기준은 질문 몇십 개로 직접 재 보고 정하세요.

## 8단계 · 대조 — 고정 길이(500자)로 잘랐다면

7주차 7-2장의 기본 청킹처럼 법령 전문을 **500자씩 고정 길이**로 잘라 같은 질문을 던집니다.
고정 길이 조각에는 조문 번호 메타데이터가 없으므로, 출처는 조각 글 안에서 **처음 나오는 「제○조(」** 를 찾아 댈 수밖에 없습니다.

In [ ]:
def law_text(law):
    """조문 전체를 한 줄기 글로 — 사람이 법령집을 읽는 순서 그대로."""
    out = []
    for u in law["조문"]["조문단위"]:
        if u["조문여부"] != "조문":
            continue
        out.append(u["조문내용"].strip())
        for h in as_list(u.get("항")):
            if h.get("항내용"):
                out.append(h["항내용"].strip())
            for ho in as_list(h.get("호")):
                out.append(ho["호내용"].strip())
                out += [m["목내용"].strip() for m in as_list(ho.get("목"))]
    return "\n".join(out)

full = law_text(cur)
SIZE = 500
fixed = [{"id": f"글자 {i}~", "text": full[i:i + ___⑮___]} for i in range(0, len(full), SIZE)]   # ⑮ 조각 길이
print(len(full), "자 →", len(fixed), "조각")
build_index("pipa_fixed", fixed)

def cite_fixed(text):
    m = re.search(r"___⑯___", text)   # ⑯ 「제○조(」 또는 「제○조의○(」
    return m.group(0) if m else "(출처 없음)"

질문 9개마다 **정답 조문**을 정해 두고, 두 방식이 1위로 고른 조문을 비교합니다.

In [ ]:
gold = {
    "정보주체의 동의 없이 개인정보를 제3자에게 제공할 수 있나요?": "제17조",
    "가명정보를 처리할 수 있는 목적은?": "제28조의2",
    "개인정보를 수집할 수 있는 경우는?": "제15조",
    "개인정보가 유출되면 정보주체에게 알려야 하나요?": "제34조",
    "공공장소에 CCTV를 설치해도 되나요?": "제25조",
    "만 14세 미만 아동의 개인정보를 처리하려면?": "제22조의2",
    "개인정보 보호책임자는 누가 지정하나요?": "제31조",
    "개인정보를 파기해야 하는 때는?": "제21조",
    "민감정보는 어떤 경우에 처리할 수 있나요?": "제23조",
}
rows = []
for q, ans in gold.items():
    art = ask(q)[0][0]
    art = re.match(r"제\d+조(?:의\d+)?", art).group(0)          # 제17조제4항 → 제17조
    fx = cite_fixed(ask(q, "pipa_fixed", k=1)[0][2])
    rows.append({"question": q[:22], "gold": ans, "article_chunk": art, "fixed_500": fx})
cmp = pd.DataFrame(rows)
cmp["article_ok"] = cmp["article_chunk"] == cmp["gold"]
cmp["fixed_ok"] = cmp["fixed_500"] == cmp["gold"]
print(cmp)
print("조문 청킹:", cmp["article_ok"].sum(), "/ 9   고정 500자:", cmp["fixed_ok"].sum(), "/ 9")

> **확인 포인트** — 법령 전문은 71,674자, 고정 길이 조각 144개입니다. 조문 청킹은 **9 / 9**, 고정 500자는 **3 / 9** 를 맞힙니다.
> 고정 길이 조각은 대개 조문 한가운데에서 시작합니다. 그래서 내용은 맞는 곳을 찾아도 조각 안에 조문 머리가 아예 없거나(「출처 없음」 5건), 처음 보이는 머리가 앞뒤의 다른 조문이 되어(제25조 질문에 제3조, 제31조 질문에 제30조의3) 엉뚱한 출처를 댑니다.

In [ ]:
assert (len(full), len(fixed)) == (71674, 144)
assert cmp["article_ok"].sum() == 9 and cmp["fixed_ok"].sum() == 3

> **Aebon's Tip** — 고정 길이에 겹침(overlap)을 주면 조금 나아지지만, 「이 조각이 몇 조 몇 항인가」라는 정보는 여전히 사라집니다. 구조가 있는 문서는 **구조대로 자르고 번호를 메타데이터로 남기는 것**이 출처 달린 RAG 의 핵심입니다.

## 9단계 · 답 생성 — 조문 번호를 인용하게

검색한 조각 3개를 `[제○조제○항]` 표시와 함께 프롬프트에 넣고, 「주어진 조문 안에서만, 번호를 인용해 답해줘」라고 요청합니다.
OpenAI 키가 있을 때만 gpt-4o-mini 를 부릅니다. 키가 없으면 검색된 근거 조문을 그대로 보여 주고 끝납니다(Colab 이면 왼쪽 열쇠 아이콘 「보안 비밀」에 `OPENAI_API_KEY` 를 넣으세요).

In [ ]:
KEY = os.environ.get("OPENAI_API_KEY")
if not KEY:
    try:
        from google.colab import userdata
        KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        KEY = None

def build_prompt(q, hits):
    ctx = "\n\n".join(f"[{cid}] {text}" for cid, _, text in hits)
    return (f"다음은 개인정보 보호법(법률 제21445호, 2026.9.11 시행) 조문입니다.\n\n{ctx}\n\n"
            f"질문: {q}\n"
            "위 조문 안에서만 근거를 찾아 3~4문장으로 답해줘. 문장마다 근거 조문을 [제○조제○항] 형식으로 붙여줘. "
            "조문에 없는 내용이면 「제공된 조문에서는 찾을 수 없습니다」라고 답해줘.")

q = "정보주체의 동의 없이 개인정보를 제3자에게 제공할 수 있나요?"
hits = ask(q)
prompt = build_prompt(q, hits)

if KEY:
    need("openai")
    from openai import OpenAI
    res = OpenAI(api_key=KEY).chat.completions.create(
        model="gpt-4o-mini", temperature=0,
        messages=[{"role": "user", "content": prompt}])
    print(res.choices[0].message.content)
else:
    print("OpenAI 키가 없어 답 생성은 건너뜁니다. 검색된 근거 조문:")
    for cid, score, text in hits:
        print(f"\n[{cid}] (유사도 {score})\n{text}")

> **확인 포인트** — 키가 없으면 근거 조문 3개가 `[제17조제4항]` 부터 나옵니다. 키가 있으면 답 문장 끝마다 `[제17조…]` 같은 인용이 붙는지 확인하세요.

In [ ]:
assert hits[0][0] == "제17조제4항"
assert "[제17조제4항]" in prompt

> **자주 겪는 오류** — LLM 이 프롬프트에 없는 조문 번호를 지어내는지 꼭 확인하세요. 답에서 `\[제\d+조` 패턴을 뽑아, 검색된 `hits` 의 id 에 없는 번호가 있으면 경고를 띄우는 것이 간단한 검증입니다.

## 10단계 · 더 해 보기

- 「개인정보 보호법 시행령」(같은 검색 결과의 두 번째 법령)을 받아 컬렉션에 함께 넣고, payload 에 `law: "법" | "시행령"` 을 두어 Qdrant 필터로 골라 검색해 보세요(7주차 7-3장 필터).
- 질문을 20개로 늘려 조문 청킹과 고정 길이 청킹의 정답률을 다시 재 보세요. `limit` 을 400 → 250 으로 줄이면 어떻게 달라지나요?
- 9단계의 인용 검증(지어낸 조문 번호 찾기)을 함수로 만들어, 답이 나올 때마다 자동으로 돌게 해 보세요.
- Streamlit(8주차)으로 질문 칸 하나 · 근거 조문 카드 셋 · 「현행 / 시행예정」 선택 단추가 있는 화면을 만들어 보세요.

## 정답

① `nw` — 현행 · 시행예정을 고르는 파라미터
② `2` — 시행예정 값
③ `법령` — 본문 JSON 의 맨 바깥 키
④ `조문` — 장 · 절 제목이 아닌 진짜 조문
⑤ `list` — 이미 목록이면 그대로
⑥ `조문가지번호` — 제7조의2 의 2
⑦ `호` — 항 아래 단계
⑧ `limit` — 길이 기준
⑨ `:memory:` — 서버 없이 메모리에
⑩ `encode` — 글을 벡터로
⑪ `768` — ko-sroberta 의 차원
⑫ `payload` — 벡터에 딸린 정보
⑬ `query_points` — Qdrant 검색 메서드
⑭ `pipa_next` — 시행예정판 컬렉션
⑮ `SIZE` — 조각 길이
⑯ `제\d+조(?:의\d+)?(?=\()` — 「제○조(」 또는 「제○조의○(」